# EEG Clinical Reasoning with LLaMA 3
Loads text file and prompts LLaMA 3 to reason like a clinician.
Runs on HPC using HuggingFace Transformers.

In [ ]:
import os
os.environ["HF_TOKEN"] = "xxxxxx"  # replace with your token

In [2]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## Load Biomarker Input

In [3]:
with open("biomarker_input.txt", "r", encoding="utf-8") as f:
    biomarker_data = f.read()

print(biomarker_data)

EEG CLINICAL REASONING REPORT

PATIENT INFORMATION
  Subject ID   : sub-007
  Dataset      : ds001
  Age          : 72
  Gender       : F
  MMSE Score   : 16

MODEL CLASSIFICATION
  Prediction   : Alzheimer's Disease
  Confidence   : 90.0%
  P(AD)        : 0.9005
  P(Control)   : 0.0995
  EEG epochs   : 383 analysed

PREDICTED EEG BIOMARKERS
  Biomarker                      Value   vs AD ref  vs CN ref  Status
  ────────────────────────────── ──────────────────────────────────────
  DTABR                          28.4123    31.1000    23.1000  NORMAL
  theta_alpha_ratio               2.6871     2.9800     1.9900  HIGH
  alpha2_abs                  7.12e-12   7.94e-12   1.82e-11  LOW
  ...

BIOMARKER RANKING (most divergent from control reference)
  1. theta_alpha_ratio: 0.70x group difference from CN  (toward AD)
  2. alpha2_abs: 0.68x group difference from CN  (toward AD)
  ...

SUMMARY FOR CLINICAL REASONING
  This patient's EEG was classified as Alzheimer's Disease with 90.0% confid

## Load LLaMA 3 Model
Set `MODEL_ID` to the model you have access to on HuggingFace.
Requires a HuggingFace token if using a gated model (e.g. `meta-llama/Meta-Llama-3-8B-Instruct`).

Run `huggingface-cli login` in the terminal first, or set `HF_TOKEN` environment variable.

In [4]:
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3" # Maybe a good model
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded on: {model.device}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded on: cuda:0


## Clinical Reasoning Prompt

In [10]:
SYSTEM_PROMPT_RESTING = """
You are a clinical neurologist specialized in EEG-based dementia diagnosis.
You reason step by step, like a clinician reviewing a patient's EEG biomarker report.

When given EEG biomarker values, you will:
1. Interpret each biomarker individually and explain what the value suggests
2. Identify whether each value is elevated, reduced, or within normal range based on known AD/FTD/CN patterns
3. Synthesize the findings across all biomarkers
4. Provide a clinical impression with a likely diagnosis (AD, FTD, or CN) and your confidence level
5. Note any inconsistencies or uncertainties in the findings

Reference patterns (resting-state EEG: eyes-open and eyes-closed combined):
- DTABR: AD ~ 31.2, FTD ~ 30.1, CN ~ 23.1 (higher = more slowing)
- Theta/Alpha Ratio: AD ~ 2.95, FTD ~ 2.69, CN ~ 1.99 (higher = more slowing)
- Alpha2 Absolute Power: CN ~ 1.82e-11, AD ~ 7.94e-12, FTD ~ 3.48e-12 (lower = more degraded)
- Alpha2 Relative Power: CN ~ 0.0376, AD ~ 0.0284, FTD ~ 0.0247 (lower = more degraded)
- Alpha2 Spectral Entropy: AD ~ 2.744, FTD ~ 2.603, CN ~ 2.609 (higher = less organized)
- Microstate B Coverage: AD ~ 0.257, FTD ~ 0.256, CN ~ 0.236 (higher = more time in microstate B)
"""

SYSTEM_PROMPT_AUDITORY = """
You are a clinical neurologist specialized in EEG-based dementia diagnosis.
You reason step by step, like a clinician reviewing a patient's EEG biomarker report.

When given EEG biomarker values, you will:
1. Interpret each biomarker individually and explain what the value suggests
2. Identify whether each value is elevated, reduced, or within normal range based on known AD/CN patterns
3. Synthesize the findings across all biomarkers
4. Provide a clinical impression with a likely diagnosis (AD or CN) and your confidence level
5. Note any inconsistencies or uncertainties in the findings

Note: These biomarkers are derived from auditory (task-based) EEG. FTD reference values are not
available for this condition. Reference patterns below are from an auditory EEG dataset and differ
from resting-state norms — do not apply resting-state reference values here.

Reference patterns (auditory EEG, AD and CN only):
- DTABR: AD ~ 0.0333, CN ~ 0.0269 (higher = more slowing)
- Theta/Alpha Ratio: AD ~ 0.00253, CN ~ 0.00184 (higher = more slowing)
- Alpha2 Absolute Power: AD ~ 0.695, CN ~ 0.622 (lower = more degraded)
- Alpha2 Relative Power: AD ~ 0.997, CN ~ 1.012 (lower = more degraded)
- Alpha2 Spectral Entropy: AD ~ 1.959, CN ~ 1.954 (nearly identical across groups)
- Microstate B Coverage: CN ~ 0.264, AD ~ 0.248 (note: CN > AD in auditory, opposite of resting-state)
"""

SYSTEM_PROMPT_UNKNOWN = """
You are a clinical neurologist specialized in EEG-based dementia diagnosis.
You reason step by step, like a clinician reviewing a patient's EEG biomarker report.

When given EEG biomarker values, you will:
1. Interpret each biomarker individually and explain what the value suggests
2. Identify the general direction of each biomarker (elevated, reduced, or typical) based on your knowledge
3. Synthesize the findings across all biomarkers
4. Provide a clinical impression with a likely diagnosis (AD, FTD, or CN) and your confidence level
5. Note any inconsistencies or uncertainties in the findings

Note: The EEG recording condition (resting-state vs task-based) is unknown. No numeric reference values
are provided because applying mismatched references could be misleading. Reason qualitatively based on
the known direction of each biomarker in dementia (e.g. increased slowing, reduced alpha power) and
clearly state that your assessment is limited by the lack of condition-matched reference values.
"""

# --- Select prompt manually based on input EEG type ---
# Options: "resting", "auditory", "unknown"
CONDITION = "resting"

SYSTEM_PROMPT = {
    "resting":  SYSTEM_PROMPT_RESTING,
    "auditory": SYSTEM_PROMPT_AUDITORY,
    "unknown":  SYSTEM_PROMPT_UNKNOWN,
}[CONDITION]

print(f"Using prompt for condition: {CONDITION}")

Using prompt for condition: resting


## Query Function

In [11]:
def ask_clinician(biomarker_data: str, question: str = None, max_new_tokens: int = 512) -> str:
    if question is None:
        question = "Please review these EEG biomarker values and provide your clinical reasoning and impression."

    user_message = f"Patient EEG Biomarker Report:\n\n{biomarker_data}\n\n{question}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## Run Clinical Reasoning

In [12]:
answer = ask_clinician(biomarker_data)
print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the EEG biomarker report, I will interpret each biomarker individually and explain what the value suggests.

1. DTABR (28.4123): This value is within the normal range for both AD and CN references. This suggests that the patient's brain activity is not significantly slowed compared to healthy controls or AD patients.
2. Theta/Alpha Ratio (2.6871): This value is higher than the reference value for AD (2.95) and CN (1.99), indicating more slowing of brain activity. This is consistent with the diagnosis of AD.
3. Alpha2 Absolute Power (7.12e-12): This value is lower than the reference value for CN (1.82e-11) and AD (7.94e-12), indicating more degraded brain activity. This is also consistent with the diagnosis of AD.
4. Alpha2 Relative Power (not provided): This biomarker is not included in the report, so I will not be able to interpret it.
5. Alpha2 Spectral Entropy (not provided): This biomarker is not included in the report, so I will not be able to interpret it.
6. Microstate 

## Custom Question

In [ ]:
answer = ask_clinician(
    biomarker_data,
    question="Which biomarker is most indicative of the diagnosis and why?"
)
print(answer)